# BNSS 2023 to Akoma Ntoso Conversion

This notebook converts the Bharatiya Nagarik Suraksha Sanhita (BNSS) 2023 PDF to Laws.Africa plaintext markup format.

BNSS replaces the Criminal Procedure Code (CrPC) 1973.

In [ ]:
import pdfplumber
import os
import re
from datetime import datetime
from langchain_azure_ai.chat_models import AzureAIChatCompletionsModel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from tqdm.notebook import tqdm

load_dotenv()

# Model info for dataset documentation
MODEL_INFO = {
    "model_id": "Llama-3.3-70B-Instruct",
    "provider": "Azure AI Foundry (Llama-3.3-70B-Instruct)",
    "temperature": 0,
    # Azure AI Foundry limits for Llama 3.3 70B:
    # - Context window: 8,192 tokens (input + output combined)
    # - max_tokens: 4,096 (default), can be 8,192 with proper deployment
    # - Rate limit: Depends on tier (S0 tier has lower limits)
    "max_tokens": 4096,  # Fixed from 120000 - Azure AI default limit
    "context_window": 8192,
    "conversion_date": datetime.now().isoformat()
}

# Azure AI Foundry configuration
# Set these environment variables:
# AZURE_INFERENCE_ENDPOINT=https://<resource>.services.ai.azure.com/models
# AZURE_INFERENCE_CREDENTIAL=<your-api-key>

print(f"Model: {MODEL_INFO['model_id']}")
print(f"Max output tokens: {MODEL_INFO['max_tokens']}")
print(f"Context window: {MODEL_INFO['context_window']}")

Model: Llama-3.3-70B-Instruct
Max output tokens: 4096
Context window: 8192


In [ ]:
def extract_pdf_text(pdf_path: str) -> str:
    """Extract text from BNSS PDF"""
    full_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text.append(text)
    return "\n".join(full_text)

raw_text = extract_pdf_text("input_pdf.pdf")
print(f"Extracted {len(raw_text)} characters from PDF")

Extracted 786357 characters from PDF


In [ ]:
with open("output/bnss_2023_text.txt", "w") as text_file:
    text_file.write(raw_text)

In [ ]:
def parse_toc(lines: list) -> tuple:
    """
    Parse the Table of Contents to extract chapter names and section names.

    Format:
    - CHAPTER <ROMAN NUMERAL>
    - <CHAPTER TITLE IN CAPS> (next line)
    - <num>. <Section name>.
    """
    chapters = []
    sections = []

    i = 0
    while i < len(lines):
        line = lines[i].strip()

        # Match chapter: "CHAPTER I", "CHAPTER II", "CHAPTER IA", etc.
        chapter_match = re.match(r'^CHAPTER\s+([IVXLCA]+)$', line)
        if chapter_match:
            roman = chapter_match.group(1)
            # Next line is the chapter title
            if i + 1 < len(lines):
                title = lines[i + 1].strip()
                # Skip lines that are just "SECTIONS" or page numbers
                if title and title != "SECTIONS" and not title.isdigit():
                    chapters.append({
                        "roman": roman,
                        "heading": title
                    })
            i += 2
            continue

        # Match section: "1. Title and extent of operation of the Code."
        section_match = re.match(r'^(\d+[A-Za-z]?)\.\s+(.+?)\.?$', line)
        if section_match:
            sec_num = section_match.group(1)
            sec_name = section_match.group(2).strip().rstrip('.')
            sections.append({
                "num": sec_num,
                "heading": sec_name
            })

        i += 1

    return chapters, sections

# Read the file and split into lines
with open("output/bnss_2023_text.txt", "r", encoding="utf-8") as f:
    all_lines = f.readlines()

# Parse TOC - adjust this number after inspecting the file
# You'll need to determine where the TOC ends in BNSS2023
toc_lines = all_lines[:800]  # Adjust this number
chapters, section_names = parse_toc(toc_lines)

print(f"Found {len(chapters)} chapters from TOC")
print(f"Found {len(section_names)} section names from TOC")

# Show first few
print("\nChapters:")
for ch in chapters[:10]:
    print(f"  CHAP {ch['roman']} - {ch['heading']}")

print("\nSections (first 10):")
for sec in section_names[:10]:
    print(f"  {sec['num']}. {sec['heading']}")

Found 40 chapters from TOC
Found 537 section names from TOC

Chapters:
  CHAP I - PRELIMINARY
  CHAP II - CONSTITUTION OF CRIMINAL COURTS AND OFFICES
  CHAP III - POWER OF COURTS
  CHAP IV - POWERS OF SUPERIOR OFFICERS OF POLICE AND AID TO THE MAGISTRATES AND THE POLICE
  CHAP V - ARREST OF PERSONS
  CHAP VI - PROCESSES TO COMPEL APPEARANCE
  CHAP VII - PROCESSES TO COMPEL THE PRODUCTION OF THINGS
  CHAP VIII - RECIPROCAL ARRANGEMENTS FOR ASSISTANCE IN CERTAIN MATTERS AND PROCEDURE FOR
  CHAP IX - SECURITY FOR KEEPING THE PEACE AND FOR GOOD BEHAVIOUR
  CHAP X - ORDER FOR MAINTENANCE OF WIVES, CHILDREN AND PARENTS

Sections (first 10):
  1. Short title, extent and commencement
  2. Definitions
  3. Construction of references
  4. Trial of offences under Bharatiya Nyaya Sanhita, 2023 and other laws
  5. Saving
  6. Classes of Criminal Courts
  7. Territorial divisions
  8. Court of Session
  9. Courts of Judicial Magistrates
  10. Chief Judicial Magistrate and Additional Chief Judicial M

## Auto-Extract Chapter Ranges and Select Chapters

This cell automatically extracts all BNSS chapters and their section ranges from the TOC.

**Key Features:**
- Automatically determines section ranges for each chapter
- Allows processing specific chapters to save costs and avoid rate limits
- No need to manually maintain chapter ranges

**Usage:**
- The next cell will show all detected chapters
- You can then select which chapters to process in the following cell

In [ ]:
# OPTIONAL: View all chapters with section counts
print("=" * 90)
print("Complete BNSS Chapter List")
print("=" * 90)
print()

total_sections = 0
for ch in CHAPTER_RANGES:
    sec_count = ch["end"] - ch["start"] + 1
    total_sections += sec_count
    print(f"CHAP {ch['roman']:6s} │ Sections {ch['start']:3d}-{ch['end']:3d} ({sec_count:3d} secs) │ {ch['heading']}")

print()
print("=" * 90)
print(f"Total: {len(CHAPTER_RANGES)} chapters, {total_sections} sections")
print("=" * 90)

Complete BNSS Chapter List

CHAP I      │ Sections   1-  5 (  5 secs) │ PRELIMINARY
CHAP II     │ Sections   6- 20 ( 15 secs) │ CONSTITUTION OF CRIMINAL COURTS AND OFFICES
CHAP III    │ Sections  21- 29 (  9 secs) │ POWER OF COURTS
CHAP IV     │ Sections  30- 34 (  5 secs) │ POWERS OF SUPERIOR OFFICERS OF POLICE AND AID TO THE MAGISTRATES AND THE POLICE
CHAP V      │ Sections  35- 62 ( 28 secs) │ ARREST OF PERSONS
CHAP VI     │ Sections  63- 93 ( 31 secs) │ PROCESSES TO COMPEL APPEARANCE
CHAP VII    │ Sections  94-110 ( 17 secs) │ PROCESSES TO COMPEL THE PRODUCTION OF THINGS
CHAP VIII   │ Sections 111-124 ( 14 secs) │ RECIPROCAL ARRANGEMENTS FOR ASSISTANCE IN CERTAIN MATTERS AND PROCEDURE FOR
CHAP IX     │ Sections 125-143 ( 19 secs) │ SECURITY FOR KEEPING THE PEACE AND FOR GOOD BEHAVIOUR
CHAP X      │ Sections 144-147 (  4 secs) │ ORDER FOR MAINTENANCE OF WIVES, CHILDREN AND PARENTS
CHAP XI     │ Sections 148-167 ( 20 secs) │ MAINTENANCE OF PUBLIC ORDER AND TRANQUILLITY
CHAP XII    │ 

In [ ]:
# Auto-extract BNSS chapter ranges from TOC
def extract_chapter_ranges_from_toc(toc_lines: list, section_names: list) -> list:
    """
    Automatically extract chapter ranges based on sections found in TOC.
    
    Args:
        toc_lines: Lines from the TOC
        section_names: List of section dicts with 'num' and 'heading'
        
    Returns:
        List of chapter dictionaries with start/end section numbers
    """
    chapter_ranges = []
    current_chapter = None
    chapter_sections = []
    
    i = 0
    while i < len(toc_lines):
        line = toc_lines[i].strip()
        
        # Match chapter header
        chapter_match = re.match(r'^CHAPTER\s+([IVXLCA]+)$', line)
        if chapter_match:
            # Save previous chapter if exists
            if current_chapter and chapter_sections:
                current_chapter["start"] = min(chapter_sections)
                current_chapter["end"] = max(chapter_sections)
                chapter_ranges.append(current_chapter)
                chapter_sections = []
            
            # Start new chapter
            roman = chapter_match.group(1)
            if i + 1 < len(toc_lines):
                heading = toc_lines[i + 1].strip()
                if heading and heading != "SECTIONS" and not heading.isdigit():
                    current_chapter = {
                        "roman": roman,
                        "heading": heading
                    }
            i += 2
            continue
        
        # Match section number
        section_match = re.match(r'^(\d+[A-Za-z]?)\.\s+', line)
        if section_match and current_chapter:
            sec_num = section_match.group(1)
            num_match = re.match(r'(\d+)', sec_num)
            if num_match:
                chapter_sections.append(int(num_match.group(1)))
        
        i += 1
    
    # Add last chapter
    if current_chapter and chapter_sections:
        current_chapter["start"] = min(chapter_sections)
        current_chapter["end"] = max(chapter_sections)
        chapter_ranges.append(current_chapter)
    
    return chapter_ranges


def filter_sections_by_chapters(sections: list, chapter_ranges: list, 
                                  selected_chapters: list = None) -> list:
    """
    Filter sections to only include specified chapters.
    
    Args:
        sections: List of section dicts
        chapter_ranges: List of chapter range dicts
        selected_chapters: List of chapter roman numerals to include (e.g., ["I", "II", "V"])
                          If None, includes all chapters
    
    Returns:
        Filtered list of sections with chapter info added
    """
    def get_chapter_for_section(section_num: str) -> dict:
        """Get chapter info for a section number."""
        try:
            num_match = re.match(r'(\d+)', section_num)
            if not num_match:
                return {"roman": "NA", "heading": "Unknown"}
            
            num = int(num_match.group(1))
            
            for chapter in chapter_ranges:
                if chapter["start"] <= num <= chapter["end"]:
                    return {"roman": chapter["roman"], "heading": chapter["heading"]}
            
            return {"roman": "NA", "heading": "Unknown"}
        except (ValueError, KeyError):
            return {"roman": "NA", "heading": "Unknown"}
    
    filtered = []
    skipped = []
    
    for sec in sections:
        chapter = get_chapter_for_section(sec["num"])
        
        # If selected_chapters is None, include all valid chapters
        # Otherwise, only include chapters in the list
        if chapter["roman"] != "NA":
            if selected_chapters is None or chapter["roman"] in selected_chapters:
                sec["chapter_roman"] = chapter["roman"]
                sec["chapter_heading"] = chapter["heading"]
                filtered.append(sec)
            else:
                skipped.append(sec["num"])
        else:
            skipped.append(sec["num"])
    
    if skipped and selected_chapters:
        print(f"Skipped {len(skipped)} sections (first 10): {skipped[:10]}")
    elif skipped:
        print(f"Skipped {len(skipped)} sections without valid chapters (first 10): {skipped[:10]}")
    
    return filtered


# Extract chapter ranges automatically from TOC
CHAPTER_RANGES = extract_chapter_ranges_from_toc(toc_lines, section_names)

print(f"Extracted {len(CHAPTER_RANGES)} chapters from TOC:\n")
for ch in CHAPTER_RANGES[:10]:  # Show first 10
    sec_count = ch["end"] - ch["start"] + 1
    print(f"  CHAP {ch['roman']:6s} - {ch['heading'][:55]:55s} (Secs {ch['start']:3d}-{ch['end']:3d})")

if len(CHAPTER_RANGES) > 10:
    print(f"  ... and {len(CHAPTER_RANGES) - 10} more chapters")

print(f"\n📝 To process specific chapters only, set:")
print(f"   selected_chapters = ['I', 'II', 'IV']  # Example")
print(f"   sections = filter_sections_by_chapters(sections, CHAPTER_RANGES, selected_chapters)")
print(f"\n📝 To process ALL chapters, set:")
print(f"   sections = filter_sections_by_chapters(sections, CHAPTER_RANGES, None)")

Extracted 40 chapters from TOC:

  CHAP I      - PRELIMINARY                                             (Secs   1-  5)
  CHAP II     - CONSTITUTION OF CRIMINAL COURTS AND OFFICES             (Secs   6- 20)
  CHAP III    - POWER OF COURTS                                         (Secs  21- 29)
  CHAP IV     - POWERS OF SUPERIOR OFFICERS OF POLICE AND AID TO THE MA (Secs  30- 34)
  CHAP V      - ARREST OF PERSONS                                       (Secs  35- 62)
  CHAP VI     - PROCESSES TO COMPEL APPEARANCE                          (Secs  63- 93)
  CHAP VII    - PROCESSES TO COMPEL THE PRODUCTION OF THINGS            (Secs  94-110)
  CHAP VIII   - RECIPROCAL ARRANGEMENTS FOR ASSISTANCE IN CERTAIN MATTE (Secs 111-124)
  CHAP IX     - SECURITY FOR KEEPING THE PEACE AND FOR GOOD BEHAVIOUR   (Secs 125-143)
  CHAP X      - ORDER FOR MAINTENANCE OF WIVES, CHILDREN AND PARENTS    (Secs 144-147)
  ... and 30 more chapters

📝 To process specific chapters only, set:
   selected_chapters = ['I'

In [ ]:
# Extract section content from the PDF text
def extract_section_content(full_text: str, section_names: list) -> list:
    """
    Extract section content from the main body text using
    the section names from TOC as anchors.

    Content format: "<num>. <Section heading>.—<content>"
    """
    sections_with_content = []

    for i, sec in enumerate(section_names):
        sec_num = sec["num"]
        sec_heading = sec["heading"]

        # Escape special regex chars in heading
        escaped_heading = re.escape(sec_heading)

        # If there's a next section available, stop at its number.
        # Otherwise, allow it to match until the end of the document (\Z).
        if i + 1 < len(section_names):
            next_sec_num = section_names[i + 1]["num"]
            # Build pattern to stop at the next section number
            stop_pattern = rf'\n{next_sec_num}\.'
        else:
            # If this is the last section, stop at the end of the document
            stop_pattern = r'\Z'

        # Pattern: section number, heading, then content until the next section or end
        pattern = rf'{sec_num}\.\s+{escaped_heading}[.\s]*[—\-–]\s*(.+?)(?={stop_pattern})'

        match = re.search(pattern, full_text, re.DOTALL)
        if match:
            content = match.group(1).strip()
            # Clean up footnote references and page numbers
            content = re.sub(r'\n\d+\n', '\n', content)  # Remove page numbers
            content = re.sub(r'\d+\[', '[', content)  # Clean footnote refs

            sections_with_content.append({
                "num": sec_num,
                "heading": sec_heading,
                "content": content
            })
        else:
            # Section might be repealed/omitted - still add with empty content
            sections_with_content.append({
                "num": sec_num,
                "heading": sec_heading,
                "content": "[Repealed/Omitted]" if "Repealed" in sec_heading or "Omitted" in sec_heading else ""
            })
    return sections_with_content


# Join the content lines (after TOC - adjust 800 to match toc_lines count)
content_text = "".join(all_lines[800:])

# Extract content for each section
sections = extract_section_content(content_text, section_names)

print(f"📄 Extracted content for {len(sections)} sections")

# Show a sample
if sections:
    print("\n✅ Sample section:")
    print(f"   Section {sections[0]['num']}: {sections[0]['heading']}")
    print(f"   Content preview: {sections[0]['content'][:150]}...")
else:
    print("\n⚠️  Warning: No sections extracted!")

📄 Extracted content for 537 sections

✅ Sample section:
   Section 1: Short title, extent and commencement
   Content preview: ...


In [ ]:
# SELECT CHAPTERS TO PROCESS
# Set selected_chapters to specific chapters you want to process, or None for all

# Option 1: Process specific chapters only (recommended for testing/rate limits)
# selected_chapters = ["I", "II"]  # Example: Process only Chapters I and II

# Option 2: Process ALL chapters (uncomment to use)
selected_chapters = None

print(f"🎯 Processing chapters: {selected_chapters if selected_chapters else 'ALL'}\n")

# Filter sections based on selection
sections = filter_sections_by_chapters(sections, CHAPTER_RANGES, selected_chapters)

# Deduplicate by section number
seen_nums = set()
unique_sections = []
for sec in sections:
    if sec["num"] not in seen_nums:
        seen_nums.add(sec["num"])
        unique_sections.append(sec)
sections = unique_sections

print(f"\n✅ {len(sections)} sections ready for processing")

# Show summary by chapter
if sections:
    print("\n📊 Sections by Chapter:")
    current_chap = None
    chap_count = 0
    for sec in sections:
        if sec.get("chapter_roman") != current_chap:
            if current_chap:
                print(f"     {chap_count} sections")
            current_chap = sec.get("chapter_roman")
            chap_count = 0
            print(f"  CHAP {current_chap} - {sec.get('chapter_heading', 'Unknown')[:50]}")
        chap_count += 1
    if chap_count:
        print(f"     {chap_count} sections")

🎯 Processing chapters: ALL


✅ 531 sections ready for processing

📊 Sections by Chapter:
  CHAP I - PRELIMINARY
     5 sections
  CHAP II - CONSTITUTION OF CRIMINAL COURTS AND OFFICES
     15 sections
  CHAP III - POWER OF COURTS
     9 sections
  CHAP IV - POWERS OF SUPERIOR OFFICERS OF POLICE AND AID TO T
     5 sections
  CHAP V - ARREST OF PERSONS
     28 sections
  CHAP VI - PROCESSES TO COMPEL APPEARANCE
     31 sections
  CHAP VII - PROCESSES TO COMPEL THE PRODUCTION OF THINGS
     17 sections
  CHAP VIII - RECIPROCAL ARRANGEMENTS FOR ASSISTANCE IN CERTAIN 
     14 sections
  CHAP IX - SECURITY FOR KEEPING THE PEACE AND FOR GOOD BEHAVI
     19 sections
  CHAP X - ORDER FOR MAINTENANCE OF WIVES, CHILDREN AND PAREN
     4 sections
  CHAP XI - MAINTENANCE OF PUBLIC ORDER AND TRANQUILLITY
     20 sections
  CHAP XII - PREVENTIVE ACTION OF THE POLICE
     5 sections
  CHAP XIII - INFORMATION TO THE POLICE AND THEIR POWERS TO INVE
     24 sections
  CHAP XIV - JURISDICTION OF THE CRIM

In [ ]:
llm = AzureAIChatCompletionsModel(
    endpoint=os.environ["AZURE_INFERENCE_ENDPOINT"],
    credential=os.environ["AZURE_INFERENCE_CREDENTIAL"],
    model=MODEL_INFO["model_id"],
    temperature=MODEL_INFO["temperature"],
    max_tokens=MODEL_INFO["max_tokens"]
)

CONVERSION_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are converting Bharatiya Nagarik Suraksha Sanhita (BNSS) 2023 sections to Laws.Africa plaintext markup format.

MARKUP RULES:
1. Section format: SEC [num]. - [heading]
2. Indent content under sections with 2 spaces
3. Subsections: SUBSEC (1), SUBSEC (2), etc. - indented under SEC
4. Paragraphs within subsections: PARA (a), PARA (b), etc.
5. Subparagraphs: SUBPARA (i), SUBPARA (ii), etc.
6. Explanations: Start with "Explanation.—" as a separate indented paragraph
7. Illustrations: Start with "Illustration" or "(a)", "(b)" as separate indented paragraphs
8. Exceptions: Start with "Exception" as a separate indented paragraph
9. Provisos: Start with "Provided that" as a separate indented paragraph

EXAMPLE OUTPUT:
```
SEC 41. - When police may arrest without warrant
  SUBSEC (1)
    Any police officer may without an order from a Magistrate and without a warrant, arrest any person—

    PARA (a)
      who commits, in the presence of a police officer, a cognizable offence;

    PARA (b)
      against whom a reasonable complaint has been made...

  SUBSEC (2)
    Any officer in charge of a police station may...

  Explanation.—For the purposes of this section...
```

PRESERVE the exact legal text. Do not paraphrase or summarize."""),
    ("human", """Convert this BNSS section to Laws.Africa markup:

Section Number: {section_num}
Section Heading: {section_heading}
Section Content:
{section_content}

Output ONLY the markup, no explanations:""")
])

chain = CONVERSION_PROMPT | llm | StrOutputParser()
print("LLM and prompt chain initialized successfully")
print(f"Max tokens per request: {MODEL_INFO['max_tokens']}")

LLM and prompt chain initialized successfully
Max tokens per request: 4096


/var/folders/5v/2b41jxz917x6mky4p2dsz1q00000gn/T/ipykernel_26389/2911067459.py:1: ExperimentalWarning: AzureAIChatCompletionsModel is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  llm = AzureAIChatCompletionsModel(


In [ ]:
# Test Azure AI Connection (Optional)
def test_azure_ai_connection():
    """Test that Azure AI is configured correctly and the model is available"""
    try:
        test_result = llm.invoke("Say 'OK' if you can read this.")
        print(f"Azure AI connection successful!")
        print(f"Model: {MODEL_INFO['model_id']}")
        print(f"Response: {test_result.content[:100]}...")
        return True
    except Exception as e:
        print(f"Azure AI connection failed: {e}")
        print("\nTroubleshooting:")
        print("1. Ensure AZURE_INFERENCE_ENDPOINT is set correctly in .env")
        print("2. Ensure AZURE_INFERENCE_CREDENTIAL (API key) is set correctly in .env")
        print(f"3. Verify model '{MODEL_INFO['model_id']}' is deployed in your Azure AI resource")
        print("4. Check that your Azure AI resource is active and not suspended")
        return False

test_azure_ai_connection()

Azure AI connection successful!
Model: Llama-3.3-70B-Instruct
Response: OK...


True

In [ ]:
import time
import json
from pathlib import Path

# Rate Limiting Configuration for Azure AI S0 Tier
RATE_LIMIT_CONFIG = {
    "delay_between_requests": 5,
    "batch_size": 3,
    "batch_delay": 30,
    "max_retries": 3,
    "initial_backoff": 60,
    "checkpoint_file": "output/bnss_conversion_checkpoint.json"
}

def load_checkpoint():
    """Load checkpoint if it exists"""
    checkpoint_path = Path(RATE_LIMIT_CONFIG["checkpoint_file"])
    if checkpoint_path.exists():
        with open(checkpoint_path, "r") as f:
            return json.load(f)
    return None

def save_checkpoint(last_index, results, total_sections):
    """Save checkpoint to file"""
    checkpoint = {
        "last_completed_index": last_index,
        "completed_sections": results,
        "timestamp": datetime.now().isoformat(),
        "total_sections": total_sections
    }
    with open(RATE_LIMIT_CONFIG["checkpoint_file"], "w") as f:
        json.dump(checkpoint, f, indent=2)

def process_all_sections(sections: list, batch_size: int = None, delay: float = None, 
                         resume: bool = True) -> tuple:
    """
    Process all sections with checkpointing and timeout handling.
    
    Args:
        sections: List of section dictionaries
        batch_size: Override batch_size from config (optional)
        delay: Override delay between requests (optional)
        resume: Auto-resume from checkpoint if exists (default: True)
    """
    results = []
    errors = []
    start_index = 0
    
    batch_size = batch_size or RATE_LIMIT_CONFIG["batch_size"]
    delay = delay or RATE_LIMIT_CONFIG["delay_between_requests"]
    
    # Check for existing checkpoint
    if resume:
        checkpoint = load_checkpoint()
        if checkpoint:
            print(f"📂 Found checkpoint from {checkpoint['timestamp']}")
            print(f"   Last completed: Section {checkpoint['last_completed_index'] + 1}/{checkpoint['total_sections']}")
            
            response = input("\n   Resume from checkpoint? (y/n): ").strip().lower()
            if response == 'y':
                results = checkpoint["completed_sections"]
                start_index = checkpoint["last_completed_index"] + 1
                print(f"✅ Resuming from section {start_index + 1}")
            else:
                print("🔄 Starting fresh (checkpoint will be overwritten)")
    
    print(f"\n⚙️  Rate Limiting Settings:")
    print(f"   - Delay between requests: {delay}s")
    print(f"   - Batch size: {batch_size} sections")
    print(f"   - Batch cooldown: {RATE_LIMIT_CONFIG['batch_delay']}s")
    print(f"   - Max tokens per request: {MODEL_INFO['max_tokens']}")
    print(f"\n🚀 Processing sections {start_index + 1} to {len(sections)} ({len(sections) - start_index} sections)...\n")

    # Track timing for ETA
    section_times = []
    
    for i in range(start_index, len(sections)):
        section = sections[i]
        section_start_time = time.time()
        retry_count = 0
        success = False
        
        while retry_count < RATE_LIMIT_CONFIG["max_retries"] and not success:
            try:
                result = chain.invoke({
                    "section_num": section["num"],
                    "section_heading": section["heading"],
                    "section_content": section["content"]
                })
                
                results.append({
                    "num": section["num"],
                    "markup": result
                })
                
                success = True
                section_time = time.time() - section_start_time
                section_times.append(section_time)
                
                # Calculate ETA
                if len(section_times) > 0:
                    avg_time = sum(section_times) / len(section_times)
                    remaining = len(sections) - (i + 1)
                    eta_seconds = remaining * avg_time
                    eta_minutes = eta_seconds / 60
                    progress_pct = ((i + 1) / len(sections)) * 100
                    
                    print(f"✓ Section {section['num']}: {section['heading'][:60]}... "
                          f"({i + 1}/{len(sections)} - {progress_pct:.1f}%, ETA: {eta_minutes:.1f}m)")
                else:
                    print(f"✓ Section {section['num']}: {section['heading'][:60]}...")
                
                # Wait between requests
                time.sleep(delay)
                
                # Save checkpoint and longer pause after each batch
                if (i + 1) % batch_size == 0:
                    save_checkpoint(i, results, len(sections))
                    print(f"\n💾 Checkpoint saved ({i + 1}/{len(sections)} sections)")
                    print(f"⏸️  Batch {(i + 1)//batch_size} complete")
                    print(f"   Cooling down for {RATE_LIMIT_CONFIG['batch_delay']} seconds...\n")
                    time.sleep(RATE_LIMIT_CONFIG['batch_delay'])

            except Exception as e:
                error_str = str(e)
                
                # Check for retryable errors (rate limit, timeout, connection)
                retryable_errors = [
                    "429", "rate limit",           # Rate limiting
                    "ReadTimeout", "read timed out",  # Timeouts
                    "ConnectionError", "Connection reset"  # Connection issues
                ]
                is_retryable = any(err.lower() in error_str.lower() for err in retryable_errors)
                
                if is_retryable and retry_count < RATE_LIMIT_CONFIG["max_retries"]:
                    retry_count += 1
                    wait_time = RATE_LIMIT_CONFIG["initial_backoff"] * (2 ** (retry_count - 1))
                    
                    error_type = "Rate limit" if "429" in error_str else "Timeout" if "timeout" in error_str.lower() else "Connection error"
                    print(f"\n⚠️  {error_type} on section {section['num']}")
                    print(f"   Attempt {retry_count}/{RATE_LIMIT_CONFIG['max_retries']}")
                    print(f"   Waiting {wait_time} seconds before retry...\n")
                    
                    time.sleep(wait_time)
                else:
                    # Non-retryable error or max retries exceeded
                    print(f"\n❌ Error on section {section['num']}: {error_str[:100]}")
                    errors.append({
                        "num": section["num"],
                        "error": error_str
                    })
                    # Save checkpoint before moving on
                    save_checkpoint(i, results, len(sections))
                    print(f"💾 Checkpoint saved before skipping section {section['num']}")
                    break
        
        if not success:
            print(f"⏭️  Skipping section {section['num']} after {retry_count} retries\n")

    # Final checkpoint save
    save_checkpoint(len(sections) - 1, results, len(sections))
    print(f"\n💾 Final checkpoint saved")
    
    return results, errors

# Start processing with checkpoint support
converted_sections, processing_errors = process_all_sections(sections, resume=True)

print(f"\n{'='*60}")
print(f"✅ Successfully converted: {len(converted_sections)} sections")
print(f"❌ Errors: {len(processing_errors)} sections")

if processing_errors:
    print(f"\n⚠️  Sections with errors:")
    for err in processing_errors[:5]:
        print(f"   - Section {err['num']}: {err['error'][:80]}")

# Merge chapter info from sections into converted_sections
for conv_sec in converted_sections:
    for orig_sec in sections:
        if orig_sec["num"] == conv_sec["num"]:
            conv_sec["chapter_roman"] = orig_sec["chapter_roman"]
            conv_sec["chapter_heading"] = orig_sec["chapter_heading"]
            break

print(f"\n📊 Chapter info merged for {len(converted_sections)} sections")
print(f"\n💡 Tip: If interrupted, just re-run this cell to resume from checkpoint!")


⚙️  Rate Limiting Settings:
   - Delay between requests: 5s
   - Batch size: 3 sections
   - Batch cooldown: 30s
   - Max tokens per request: 4096

🚀 Processing sections 1 to 531 (531 sections)...

✓ Section 1: Short title, extent and commencement... (1/531 - 0.2%, ETA: 12.6m)
✓ Section 2: Definitions... (2/531 - 0.4%, ETA: 41.4m)
✓ Section 3: Construction of references... (3/531 - 0.6%, ETA: 30.2m)

💾 Checkpoint saved (3/531 sections)
⏸️  Batch 1 complete
   Cooling down for 30 seconds...

✓ Section 4: Trial of offences under Bharatiya Nyaya Sanhita, 2023 and ot... (4/531 - 0.8%, ETA: 25.5m)
✓ Section 5: Saving... (5/531 - 0.9%, ETA: 21.4m)
✓ Section 6: Classes of Criminal Courts... (6/531 - 1.1%, ETA: 20.4m)

💾 Checkpoint saved (6/531 sections)
⏸️  Batch 2 complete
   Cooling down for 30 seconds...

✓ Section 7: Territorial divisions... (7/531 - 1.3%, ETA: 20.0m)
✓ Section 8: Court of Session... (8/531 - 1.5%, ETA: 27.5m)
✓ Section 9: Courts of Judicial Magistrates... (9/531 - 1.7%,

## Process Sections with Checkpoint Support

This cell processes all selected sections and converts them to Akoma Ntoso markup.

**Features:**
- ✅ **Auto-resume**: Automatically resumes from last checkpoint if interrupted
- ✅ **Timeout handling**: Retries on timeouts and connection errors (not just rate limits)
- ✅ **Progress tracking**: Shows ETA and completion percentage
- ✅ **Checkpoint saving**: Saves progress after every batch (3 sections)
- ✅ **Error recovery**: Skips problematic sections after max retries

**How to use:**
1. **First run**: Just run this cell - it will start from section 1
2. **If interrupted**: Re-run this cell and press 'y' to resume from checkpoint
3. **Start fresh**: Run the checkpoint management cell below and uncomment the delete line

In [ ]:
# CHECKPOINT MANAGEMENT HELPERS
# Use these cells to manage checkpoints manually

import json
from pathlib import Path

checkpoint_file = Path("output/bnss_conversion_checkpoint.json")

# View checkpoint status
if checkpoint_file.exists():
    with open(checkpoint_file, "r") as f:
        cp = json.load(f)
    print("📊 Checkpoint Status:")
    print(f"   File: {checkpoint_file}")
    print(f"   Created: {cp['timestamp']}")
    print(f"   Progress: {cp['last_completed_index'] + 1}/{cp['total_sections']} sections")
    print(f"   Completion: {((cp['last_completed_index'] + 1) / cp['total_sections'] * 100):.1f}%")
    print(f"\n📝 Actions:")
    print(f"   - To resume: Just re-run the processing cell above")
    print(f"   - To start fresh: Run the 'Delete checkpoint' cell below")
else:
    print("ℹ️  No checkpoint found")
    print("   Processing will start from beginning")

print("\n" + "="*60)

# Uncomment and run to DELETE checkpoint (start fresh)
# checkpoint_file.unlink(missing_ok=True)
# print("🗑️  Checkpoint deleted - will start fresh on next run")

In [ ]:
# Save the converted sections to a file
output_file = "output/bnss_2023_markup.txt"

with open(output_file, "w", encoding="utf-8") as f:
    # Group by chapters
    current_chapter = None
    
    for sec in converted_sections:
        chapter_id = f"{sec.get('chapter_roman', 'NA')}"
        
        # Write chapter header if new chapter
        if chapter_id != current_chapter:
            current_chapter = chapter_id
            f.write(f"\n\nCHAPTER {sec.get('chapter_roman', 'NA')}\n")
            f.write(f"{sec.get('chapter_heading', 'Unknown')}\n")
            f.write("=" * 80 + "\n\n")
        
        # Write section markup
        f.write(sec["markup"])
        f.write("\n\n")

print(f"Markup saved to {output_file}")
print(f"Total sections: {len(converted_sections)}")
print(f"Conversion complete!")

Markup saved to output/bnss_2023_markup.txt
Total sections: 531
Conversion complete!


In [ ]:
# Optional: Save metadata
import json

metadata = {
    "document": "Bharatiya Nagarik Suraksha Sanhita 2023",
    "act_number": "46 of 2023",
    "replaces": "Criminal Procedure Code (CrPC) 1973",
    "conversion_date": MODEL_INFO["conversion_date"],
    "model": MODEL_INFO["model_id"],
    "provider": MODEL_INFO["provider"],
    "sections_converted": len(converted_sections),
    "chapters": len(set(sec.get('chapter_roman', 'NA') for sec in converted_sections)),
    "errors": len(processing_errors)
}

with open("output/bnss_2023_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Metadata saved to output/bnss_2023_metadata.json")
print(json.dumps(metadata, indent=2))